# Phase 1 - Death-risk prediction notebook walkthrough

In this notebook, we implement and compare multiple classification-based machine learning algorithms to predict death-risk categories from historical risk-factor measurements. The risk-factor CSV provides the model inputs, while official registered-death totals imported from PSA Excel workbooks provide the target signal used to label each year as `Low Risk`, `Medium Risk`, or `High Risk`.

**Disclaimer:** *The target variable in this notebook is a shifted registered-death count used for machine-learning demonstration. It combines historical risk-factor proxy totals with PSA registered-death totals from Excel for recent years, plus a provisional annualized 2025 estimate. It should not be interpreted as an individual clinical mortality-risk measure or a finalized epidemiological forecast.*

- **Phase 1**: We explain each code cell, load the risk-factor CSV and death-count Excel files, perform data cleaning and feature engineering, shift the target forward by one year to avoid leakage, and split the data.
- **Phase 2**: We train and compare Logistic Regression, SVM, Decision Tree, and Random Forest classifiers.
- **Phase 3**: We summarize model results, feature relationships, ROC behavior, and practical limitations.

### Environment setup and imports

**Purpose**: Import all Python libraries needed for data manipulation, visualization, and machine learning models (Logistic Regression, SVM, Decision Tree, Random Forest).

**Input**: No data inputs yet. This cell only brings in external packages such as `pandas`, `numpy`, `scikit-learn`, `matplotlib`, and `seaborn`.

**Output**: Loaded modules in the Python session, enabling data processing, splitting mechanisms, classification modeling, and visualization for the rest of the notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, TimeSeriesSplit, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                           roc_auc_score, roc_curve, precision_recall_curve)
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import label_binarize
from sklearn.pipeline import make_pipeline
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

### Code explanation - imports

This cell prepares the Python environment for the whole death-risk prediction workflow.

- `pandas` and `numpy` handle tabular data, missing values, numeric arrays, and simple calculations.
- `matplotlib` and `seaborn` create the charts used later for correlations, feature importance, and ROC curves.
- `train_test_split`, `GridSearchCV`, and related tools create training/testing splits and tune model settings.
- `StandardScaler` standardizes features for scale-sensitive models, while `LabelEncoder` converts risk labels into numeric classes.
- Logistic Regression, SVM, Decision Tree, and Random Forest are the four classifiers compared in this notebook.
- The metric imports allow the notebook to evaluate death-risk predictions using accuracy, ROC AUC, confusion-matrix logic, and curve-based diagnostics.


### Step 1 - Load risk factors and Excel death-count targets

**Purpose**: Load the core risk-factor dataset, filter it to the Philippines, linearly extrapolate missing recent risk-factor years (2020-2025), and import registered-death totals from PSA Excel files. The death-count target is shifted so risk factors at year `t` predict the death-risk category for year `t + 1`.

**Input**: `Number of Deaths by Risk Factors.csv` for feature columns, `2_2024 Deaths_Textual Tables.xlsx` for annual registered deaths from 2015-2024, and `Table 1_Attachment to PR on 2025 COD_as of 28 February 2026.xlsx` for a provisional 2025 death-count signal.

**Output**: `df`, a modeling DataFrame containing historical/extrapolated risk factors, `Total_Deaths_Real`, `Death_Target_Source`, and `Next_Year_Target`. The final year is removed because it has no next-year target.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ---------------------------------------------------------
# STEP A: LOAD THE RISK-FACTOR FEATURES
# ---------------------------------------------------------
RISK_FACTORS_FILE = Path('Number of Deaths by Risk Factors.csv')
ANNUAL_DEATHS_EXCEL_FILE = Path('2_2024 Deaths_Textual Tables.xlsx')
PROVISIONAL_2025_EXCEL_FILE = Path('Table 1_Attachment to PR on 2025 COD_as of 28 February 2026.xlsx')

# The CSV contains risk-attributed death counts by country/year. For this notebook,
# these columns are treated as predictors for next-year death-risk category.
df_risk = pd.read_csv(RISK_FACTORS_FILE)
df_risk = df_risk[df_risk['Entity'] == 'Philippines'].copy()
df_risk.drop(columns=['Entity', 'Code'], inplace=True)
df_risk = df_risk.sort_values('Year').reset_index(drop=True)

# Original risk factor columns (exclude 'Year')
risk_factor_cols = [c for c in df_risk.columns if c != 'Year']

# ---------------------------------------------------------
# STEP B: IMPORT OFFICIAL/PROVISIONAL DEATH TARGETS FROM EXCEL
# ---------------------------------------------------------
def load_annual_registered_deaths(workbook_path):
    """Read PSA Table 1 annual registered deaths from the textual Excel workbook."""
    table = pd.read_excel(workbook_path, sheet_name='Table1', header=None)
    year_row = table.iloc[1, 1:]
    number_row = table.iloc[2, 1:]

    annual_deaths = {}
    for year, total in zip(year_row, number_row):
        if pd.notna(year) and pd.notna(total):
            annual_deaths[int(year)] = int(total)
    return annual_deaths


def load_annualized_2025_provisional_deaths(workbook_path):
    """Read the Jan-Oct 2025 provisional total and annualize it for demo use."""
    table = pd.read_excel(workbook_path, sheet_name='Table 1', header=None)
    total_rows = table[table.iloc[:, 0].astype(str).str.strip().str.lower().eq('total')]

    if total_rows.empty:
        return None, None

    jan_oct_total = pd.to_numeric(total_rows.iloc[0, 1], errors='coerce')
    if pd.isna(jan_oct_total):
        return None, None

    annualized_total = int(round(jan_oct_total * 12 / 10))
    return annualized_total, int(jan_oct_total)


recent_death_counts = load_annual_registered_deaths(ANNUAL_DEATHS_EXCEL_FILE)
target_sources = {
    year: f'PSA annual registered deaths from {ANNUAL_DEATHS_EXCEL_FILE.name}'
    for year in recent_death_counts
}

provisional_2025_total, provisional_2025_jan_oct = load_annualized_2025_provisional_deaths(
    PROVISIONAL_2025_EXCEL_FILE
)
if provisional_2025_total is not None:
    recent_death_counts[2025] = provisional_2025_total
    target_sources[2025] = (
        f'PSA provisional Jan-Oct 2025 deaths from {PROVISIONAL_2025_EXCEL_FILE.name}, '
        'annualized for demonstration'
    )

print('Excel death-count targets loaded:')
print(pd.Series(recent_death_counts, name='Registered_Deaths').sort_index())
if provisional_2025_jan_oct is not None:
    print()
    print(f'2025 note: Jan-Oct provisional total = {provisional_2025_jan_oct:,}; annualized demo target = {provisional_2025_total:,}')

# ---------------------------------------------------------
# STEP C: EXTRAPOLATE RISK FACTORS FOR 2020 - 2025
# ---------------------------------------------------------
years_to_add = [2020, 2021, 2022, 2023, 2024, 2025]
future_df = pd.DataFrame({'Year': years_to_add})

for col in risk_factor_cols:
    poly = np.polyfit(df_risk['Year'], df_risk[col], 1)
    future_df[col] = np.polyval(poly, future_df['Year'])

df_risk_extended = pd.concat([df_risk, future_df], ignore_index=True)

# ---------------------------------------------------------
# STEP D: ALIGN THE DEATH TARGETS WITH THE RISK-FACTOR ROWS
# ---------------------------------------------------------
deaths_data = []
for year in df_risk_extended['Year']:
    year = int(year)

    if year in recent_death_counts:
        total = recent_death_counts[year]
        source = target_sources[year]
    else:
        row = df_risk[df_risk['Year'] == year]
        if not row.empty:
            total = row[risk_factor_cols].sum(axis=1).values[0]
            source = 'Risk-factor sum proxy from CSV (used before Excel annual totals are available)'
        else:
            total = np.nan
            source = 'Missing target'

    deaths_data.append({
        'Year': year,
        'Total_Deaths_Real': total,
        'Death_Target_Source': source
    })

df_targets = pd.DataFrame(deaths_data)

# ---------------------------------------------------------
# STEP E: MERGE AND CREATE THE NEXT-YEAR PREDICTION TARGET
# ---------------------------------------------------------
df_aligned = pd.merge(df_risk_extended, df_targets, on='Year', how='left')
df_aligned = df_aligned.sort_values('Year').reset_index(drop=True)

# Shift target by -1 so risk factors at year t predict deaths in year t+1
df_aligned['Next_Year_Target'] = df_aligned['Total_Deaths_Real'].shift(-1)
df_aligned['Next_Year_Target_Source'] = df_aligned['Death_Target_Source'].shift(-1)

# Drop the last row (2025) because there is no next-year target
df = df_aligned.dropna(subset=['Next_Year_Target']).copy()

# Keep a numeric reference to the target deaths for analysis
df['Total_Deaths_Real'] = df['Total_Deaths_Real'].astype(int)
df['Next_Year_Target'] = df['Next_Year_Target'].astype(int)


### Code explanation - loading risk features and death targets

This cell builds the main modeling table. It has three important jobs.

First, it loads the risk-factor CSV and filters it to the Philippines. These risk-factor columns become the input variables, or features, used by the models.

Second, it imports official or provisional registered-death totals from the Excel files. The `load_annual_registered_deaths()` helper reads annual deaths from the 2015-2024 PSA workbook, while `load_annualized_2025_provisional_deaths()` reads the provisional Jan-Oct 2025 total and annualizes it only for demonstration.

Third, it aligns the death totals with the risk-factor rows and creates `Next_Year_Target` by shifting deaths one year forward. This is important because the notebook is framed as prediction: risk factors from year `t` are used to predict the death-risk category for year `t + 1`, instead of using the same year's death total as both input context and target.

The `Death_Target_Source` and `Next_Year_Target_Source` columns are included so the notebook remains transparent about which target values came from Excel and which older values are still proxy totals from the CSV.


### Dataset quality check

**Purpose**: Confirm whether the aligned modeling table still has missing values after filtering, extrapolation, merging, and target shifting.

**Input**: The prepared DataFrame `df` from the previous step.

**Output**: A column-by-column count of missing values. A clean result here means the next preprocessing cells can safely categorize and split the data.


In [ ]:
# Quick check for missing values
print("Missing Values:")
print(df.isnull().sum())

### How to read this output - missing values

The printed table shows how many missing values exist in each column after loading, extrapolating, merging, and shifting the target. Columns with zero missing values are ready for modeling. If an important feature or target column had missing values here, the next preprocessing step would need to fill or remove those rows before training.


### Step 2 - Data cleaning and preprocessing

**Purpose**: Clean column names, structure risk categories, and correct potential target leakage by computing risk as a future-shifted forecasting objective. The model uses the original risk-factor columns as features, while the Excel-supported registered-death totals define the target labels.

**Input**: The prepared metrics in `df`, specifically the base numeric risk-factor columns and the shifted `Next_Year_Target`.

**Output**: A clean DataFrame with properly categorized target bins (`Low Risk`, `Medium Risk`, `High Risk`) and a feature list ready for model training.

In [ ]:
# Clean column names (remove leading/trailing spaces)
df.columns = df.columns.str.strip()

# Define the feature columns – only the original risk factor columns
feature_cols = [c for c in risk_factor_cols if c in df.columns]

# Create death risk categories based on next year's total deaths
percentiles = df['Next_Year_Target'].quantile([1/3, 2/3]).values

def categorize_risk(deaths):
    if deaths <= percentiles[0]:
        return 'Low Risk'
    elif deaths <= percentiles[1]:
        return 'Medium Risk'
    else:
        return 'High Risk'

df['Death_Risk_Category'] = df['Next_Year_Target'].apply(categorize_risk)

print("Death Risk Category Distribution:")
print(df['Death_Risk_Category'].value_counts())
print(f"\nPercentiles used: {percentiles}")

### Code explanation - creating death-risk categories

This cell turns the continuous `Next_Year_Target` death count into a classification label.

- `df.columns.str.strip()` removes accidental spaces from column names.
- `feature_cols` keeps only the original risk-factor columns, preventing target columns such as `Total_Deaths_Real` from leaking into the model inputs.
- The 33rd and 66th percentiles split next-year death counts into three relative groups.
- `categorize_risk()` assigns each row to `Low Risk`, `Medium Risk`, or `High Risk` based on those percentile cutoffs.

These labels are relative to this dataset. For example, `High Risk` means the next year's registered-death target is in the upper third of this notebook's target distribution, not that it is a clinical diagnosis for an individual person.


### Exploratory Data Analysis (EDA)

**Purpose**: Add a stronger final-project data exploration step before modeling. This shows the shape of the prepared dataset, the death-count target trend, the class distribution, and the recent Excel-backed target sources.

**Input**: The cleaned DataFrame `df`, the selected `feature_cols`, `Total_Deaths_Real`, `Next_Year_Target`, and `Death_Risk_Category`.

**Output**: Summary tables and charts that help explain what the model is learning from before any algorithm is trained.


In [ ]:
# --- FINAL PROJECT EDA ---
eda_columns = [
    'Year',
    'Total_Deaths_Real',
    'Next_Year_Target',
    'Death_Risk_Category',
    'Death_Target_Source',
    'Next_Year_Target_Source'
]

print(f"Prepared modeling dataset shape: {df.shape}")
print(f"Number of model features: {len(feature_cols)}")
print()
print("Preview of target-aligned rows:")
display(df[eda_columns].head(10))

print("Summary statistics for death target columns:")
display(df[['Total_Deaths_Real', 'Next_Year_Target']].describe().round(2))

print()
print("Death-risk class distribution:")
risk_order = ['Low Risk', 'Medium Risk', 'High Risk']
target_counts = df['Death_Risk_Category'].value_counts().reindex(risk_order)
display(target_counts.rename('Count').to_frame())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(df['Year'], df['Total_Deaths_Real'], marker='o', label='Current-year deaths')
axes[0].plot(df['Year'], df['Next_Year_Target'], marker='s', label='Next-year target')
axes[0].set_title('Registered/Proxy Death Target by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Deaths')
axes[0].legend()
axes[0].tick_params(axis='x', rotation=45)

sns.barplot(x=target_counts.index, y=target_counts.values, ax=axes[1])
axes[1].set_title('Death-Risk Category Balance')
axes[1].set_xlabel('Risk Category')
axes[1].set_ylabel('Rows')

axes[2].hist(df['Next_Year_Target'], bins=10, edgecolor='black', alpha=0.75)
axes[2].axvline(percentiles[0], color='red', linestyle='--', label='33rd percentile')
axes[2].axvline(percentiles[1], color='green', linestyle='--', label='66th percentile')
axes[2].set_title('Next-Year Death Target Distribution')
axes[2].set_xlabel('Next-Year Deaths')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.show()

print("Recent target-source audit:")
display(df[eda_columns].tail(12))


### Code explanation - exploratory data analysis

This section is added to make the notebook feel more like a complete final project. It proves that the model is not being trained blindly: we inspect the target columns, confirm the class balance, visualize the death-count trend, and audit which recent target values came from Excel.

The most important check is the class distribution. If one class dominated the dataset, accuracy could become misleading. Here the three classes are intentionally built from percentile cutoffs, so the labels are fairly balanced for demonstration.


### Step 3 - Split the data for training and testing

**Purpose**: Create encoded target labels and split the tiny dataset into training and testing sets. The source notebook discusses time-series validation as the ideal direction for chronological data, but with this very small sample the test folds can easily lose one or more classes. This fixed notebook uses a stratified split so each risk category remains represented during demonstration.

**Input**: `feature_cols`, the encoded risk category target, and the categorized DataFrame.

**Output**: `X_train`, `X_test`, `y_train`, and `y_test`, with class proportions preserved as much as possible.


In [ ]:
# Encode target variable
le_target = LabelEncoder()
df['Target_Encoded'] = le_target.fit_transform(df['Death_Risk_Category'])

X = df[feature_cols]
y = df['Target_Encoded']

# Stratified split preserves class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

### Code explanation - encoding and splitting

Machine-learning classifiers need numeric target labels, so `LabelEncoder` converts `Low Risk`, `Medium Risk`, and `High Risk` into integers. The feature matrix `X` contains only risk-factor inputs, while `y` contains the encoded death-risk class.

The `train_test_split()` call separates the data into training and testing rows. `stratify=y` is important because this dataset is small; it tries to preserve the same class balance in both sets so that the test set still contains examples from each death-risk category.


## Phase 2 - Model training and comparison

This phase trains four common classification algorithms and compares their performance. The models are intentionally simple or constrained because the dataset is extremely small and a complex model can memorize the historical sequence instead of learning a stable pattern.


### Feature scaling

**Purpose**: Standardize numeric risk-factor values before fitting models that are sensitive to feature scale, especially Logistic Regression and SVM.

**Input**: Raw numeric training and testing matrices.

**Output**: `X_train_scaled` and `X_test_scaled`, where each feature is centered and scaled using statistics learned only from the training set.


In [ ]:
# Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### Code explanation - feature scaling

This cell standardizes the input features. Scaling is needed because risk-factor columns can have different numeric ranges, and models such as Logistic Regression and SVM are affected by feature scale.

The scaler is fitted only on `X_train` with `fit_transform()`. The test set uses `transform()` so the model does not learn anything from the test data before evaluation.


### Logistic Regression

**Purpose**: Train a simple multinomial baseline classifier. Logistic Regression is useful here because it is less prone to extreme memorization than deeper tree-based models on tiny datasets.

**Input**: Scaled training and testing features, plus encoded risk labels.

**Output**: Predicted classes, class probabilities, accuracy, and multiclass one-vs-rest ROC AUC when it is defined.


In [ ]:
lr_model = LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr   = lr_model.predict(X_test_scaled)
y_proba_lr  = lr_model.predict_proba(X_test_scaled)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
# ROC AUC is not defined when test set does not contain all classes
try:
    lr_auc = roc_auc_score(y_test, y_proba_lr, multi_class='ovr')
except ValueError:
    lr_auc = np.nan

print(f"Logistic Regression Accuracy: {lr_accuracy:.4f}")
print(f"Logistic Regression ROC AUC: {lr_auc:.4f}")

### Interpretation guide - Logistic Regression

Logistic Regression is the baseline model. It learns linear relationships between scaled risk factors and the encoded death-risk classes. The model then outputs both predicted labels and class probabilities.

Accuracy tells how many test rows were classified correctly. ROC AUC summarizes how well the predicted probabilities separate the classes in a one-vs-rest setup. Because the dataset is small, these numbers are useful for comparing notebook experiments, but they should not be treated as final public-health evidence.


### Support Vector Machine (SVM)

**Purpose**: Train a nonlinear SVM classifier and use `GridSearchCV` to select basic RBF-kernel hyperparameters.

**Input**: Scaled feature matrices and encoded labels.

**Output**: The best SVM parameters from the grid search, predictions, probabilities, accuracy, and ROC AUC when it is defined.


In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf']
}
grid = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=3, scoring='accuracy')
grid.fit(X_train_scaled, y_train)
svm_model = grid.best_estimator_

y_pred_svm  = svm_model.predict(X_test_scaled)
y_proba_svm = svm_model.predict_proba(X_test_scaled)

svm_accuracy = accuracy_score(y_test, y_pred_svm)
try:
    svm_auc = roc_auc_score(y_test, y_proba_svm, multi_class='ovr')
except ValueError:
    svm_auc = np.nan

print(f"SVM Best Params: {grid.best_params_}")
print(f"SVM Accuracy: {svm_accuracy:.4f}")
print(f"SVM ROC AUC: {svm_auc:.4f}")

### Interpretation guide - SVM with grid search

The SVM section tests several values of `C` and `gamma` for an RBF-kernel classifier. `GridSearchCV` trains multiple SVM versions and keeps the parameter combination with the best cross-validated accuracy on the training data.

In death-risk prediction terms, this model is looking for nonlinear boundaries between the Low, Medium, and High Risk classes. The printed best parameters show which SVM settings were selected before evaluating on the test set.


### Decision Tree

**Purpose**: Train an interpretable tree-based classifier that can reveal rule-like splits in the risk-factor data. The maximum depth is capped to reduce overfitting.

**Input**: Unscaled feature matrices and encoded labels. Decision trees do not require standardized inputs.

**Output**: Predicted classes, class probabilities, accuracy, and ROC AUC when it is defined.


In [ ]:
dt_model = DecisionTreeClassifier(random_state=42, max_depth=5)
dt_model.fit(X_train, y_train)

y_pred_dt  = dt_model.predict(X_test)
y_proba_dt = dt_model.predict_proba(X_test)

dt_accuracy = accuracy_score(y_test, y_pred_dt)
try:
    dt_auc = roc_auc_score(y_test, y_proba_dt, multi_class='ovr')
except ValueError:
    dt_auc = np.nan

print(f"Decision Tree Accuracy: {dt_accuracy:.4f}")
print(f"Decision Tree ROC AUC: {dt_auc:.4f}")

### Interpretation guide - Decision Tree

The Decision Tree model learns a sequence of if-then splits on the risk-factor columns. It is easier to interpret than many models because its decisions can be traced through feature thresholds.

The `max_depth=5` limit prevents the tree from growing too deep and memorizing the tiny dataset. Its accuracy and ROC AUC show how well these rule-like splits predict the held-out death-risk labels.


### Random Forest

**Purpose**: Train an ensemble of decision trees to compare against the single tree. The forest is capped in depth and number of estimators to keep the model modest for the dataset size.

**Input**: Unscaled feature matrices and encoded labels.

**Output**: Predicted classes, class probabilities, accuracy, ROC AUC when it is defined, and feature-importance values used later.


In [ ]:
rf_model = RandomForestClassifier(n_estimators=50, random_state=42, max_depth=7)
rf_model.fit(X_train, y_train)

y_pred_rf  = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
try:
    rf_auc = roc_auc_score(y_test, y_proba_rf, multi_class='ovr')
except ValueError:
    rf_auc = np.nan

print(f"Random Forest Accuracy: {rf_accuracy:.4f}")
print(f"Random Forest ROC AUC: {rf_auc:.4f}")

### Interpretation guide - Random Forest

Random Forest combines many decision trees and averages their predictions. This usually gives more stable predictions than a single tree, but it can still overfit when the dataset is very small.

The notebook uses a capped number of trees and limited depth to keep the model controlled. The trained `rf_model` is also reused later for feature importance, ROC curves, and the prediction helper function.


### Detailed model evaluation: classification reports and confusion matrices

**Purpose**: Strengthen the model-evaluation section beyond simple Accuracy and ROC AUC. A final project should also show Precision, Recall, F1-score, support, and confusion matrices so the reader can see which classes each model handles well or poorly.

**Input**: Predictions from Logistic Regression, SVM, Decision Tree, and Random Forest, plus the true test labels `y_test`.

**Output**: A detailed metric table, full classification reports for each model, and confusion-matrix heatmaps.


In [ ]:
# --- FINAL PROJECT DETAILED EVALUATION ---
model_predictions = {
    'Logistic Regression': y_pred_lr,
    'SVM': y_pred_svm,
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf,
}

all_label_ids = np.arange(len(le_target.classes_))
class_names = le_target.classes_

evaluation_rows = []

for model_name, y_pred in model_predictions.items():
    report_dict = classification_report(
        y_test,
        y_pred,
        labels=all_label_ids,
        target_names=class_names,
        output_dict=True,
        zero_division=0
    )

    evaluation_rows.append({
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Macro Precision': report_dict['macro avg']['precision'],
        'Macro Recall': report_dict['macro avg']['recall'],
        'Macro F1': report_dict['macro avg']['f1-score'],
        'Weighted F1': report_dict['weighted avg']['f1-score'],
    })

    print('=' * 70)
    print(f'Classification Report - {model_name}')
    print('=' * 70)
    print(classification_report(
        y_test,
        y_pred,
        labels=all_label_ids,
        target_names=class_names,
        zero_division=0
    ))

model_report_summary = pd.DataFrame(evaluation_rows).sort_values('Macro F1', ascending=False)
print('Summary of detailed classification metrics:')
display(model_report_summary.round(4))

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (model_name, y_pred) in zip(axes, model_predictions.items()):
    cm = confusion_matrix(y_test, y_pred, labels=all_label_ids)
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names,
        ax=ax
    )
    ax.set_title(f'{model_name} Confusion Matrix')
    ax.set_xlabel('Predicted Label')
    ax.set_ylabel('Actual Label')

plt.tight_layout()
plt.show()


### Code explanation - classification reports and confusion matrices

The classification report breaks performance down by class. This matters because a model can have decent overall accuracy while still doing poorly on one risk category.

The confusion matrices show actual labels on the rows and predicted labels on the columns. Correct predictions appear on the diagonal. Off-diagonal values show the exact kind of mistake, such as a `High Risk` year being predicted as `Medium Risk`.


### Phase 2 - Feature importance and correlation

**Purpose**: Connect model behavior to analytical insight by identifying which risk factors are most strongly associated with the shifted death target and which features the Random Forest uses most heavily.

**Input**: The modeling DataFrame, `feature_cols`, `Next_Year_Target`, and the trained Random Forest model.

**Output**: Two bar charts: one showing the strongest absolute correlations with next-year deaths and another showing the top Random Forest feature importances.


In [ ]:
# Correlation with Next_Year_Target
df_corr = df[feature_cols + ['Next_Year_Target']].corr()
top_corr = df_corr['Next_Year_Target'].abs().sort_values(ascending=False).head(10)

plt.figure(figsize=(10,5))
top_corr.iloc[1:].plot(kind='barh')
plt.title('Top Correlated Risk Factors with Next Year Deaths')
plt.tight_layout()
plt.show()

### Code explanation - correlation chart

This chart checks which risk-factor columns have the strongest absolute correlation with `Next_Year_Target`. Correlation is not the same as causation, but it helps identify which variables move most closely with the shifted death-count target.

The code drops the target itself from the bar chart by using `top_corr.iloc[1:]`, then plots the next strongest correlated features.


In [ ]:
# Random Forest feature importances
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(10,5))
importances.head(10).plot(kind='barh')
plt.title('Random Forest Feature Importances')
plt.tight_layout()
plt.show()

### Code explanation - Random Forest feature importance

Feature importance estimates which risk-factor columns the Random Forest used most when making splits. Higher importance means the feature contributed more to the ensemble's prediction decisions.

This plot complements the correlation chart: correlation describes direct linear association with the target, while Random Forest importance describes usefulness inside the trained model.


## Phase 3 - Model comparison

**Purpose**: Summarize and compare the performance across all implemented classification algorithms.

**Input**: Accuracy and ROC AUC values collected from Logistic Regression, SVM, Decision Tree, and Random Forest.

**Output**: A compact comparison table that makes it easier to see which model performed best in this demonstration run.


In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'SVM', 'Decision Tree', 'Random Forest'],
    'Accuracy': [lr_accuracy, svm_accuracy, dt_accuracy, rf_accuracy],
    'ROC AUC': [lr_auc, svm_auc, dt_auc, rf_auc]
})
print(results)

### How to read this output - model comparison table

This table places the four models side by side. `Accuracy` reports the share of correct test predictions, while `ROC AUC` reports probability-separation quality across the death-risk classes.

A higher value is better, but the dataset is small enough that one or two rows can strongly change the ranking. The safest conclusion is comparative: which algorithm worked better in this demonstration pipeline, not which one is ready for real deployment.


### Cross-validation for model stability

**Purpose**: Add a stronger validation check so the project does not rely only on one train-test split. Because this is a very small multiclass dataset, cross-validation helps show whether model performance is stable across several resampled folds.

**Input**: Full feature matrix `X`, encoded target `y`, and the same four model families used in the main comparison.

**Output**: Fold-level accuracy scores, mean accuracy, and standard deviation for each model.


In [ ]:
# --- FINAL PROJECT CROSS-VALIDATION ---
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    'Logistic Regression': make_pipeline(
        StandardScaler(),
        LogisticRegression(solver='lbfgs', max_iter=1000, random_state=42)
    ),
    'SVM': make_pipeline(
        StandardScaler(),
        SVC(
            probability=True,
            random_state=42,
            C=grid.best_params_['C'],
            gamma=grid.best_params_['gamma'],
            kernel=grid.best_params_['kernel']
        )
    ),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=5),
    'Random Forest': RandomForestClassifier(n_estimators=50, random_state=42, max_depth=7),
}

cv_rows = []
for model_name, model in cv_models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    row = {
        'Model': model_name,
        'Mean CV Accuracy': scores.mean(),
        'Std CV Accuracy': scores.std(),
        'Min Fold Accuracy': scores.min(),
        'Max Fold Accuracy': scores.max(),
    }
    for fold_idx, score in enumerate(scores, start=1):
        row[f'Fold {fold_idx}'] = score
    cv_rows.append(row)

cv_results = pd.DataFrame(cv_rows).sort_values('Mean CV Accuracy', ascending=False)
print('5-Fold Stratified Cross-Validation Accuracy:')
display(cv_results.round(4))

plt.figure(figsize=(11, 6))
plt.barh(
    cv_results['Model'],
    cv_results['Mean CV Accuracy'],
    xerr=cv_results['Std CV Accuracy'],
    color='steelblue',
    alpha=0.8,
    capsize=5
)
plt.xlabel('Mean CV Accuracy')
plt.title('5-Fold Stratified Cross-Validation Results')
plt.xlim(0, 1)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()


### Code explanation - cross-validation

This section uses `StratifiedKFold` because the target has three classes and the dataset is small. Stratification keeps the Low, Medium, and High Risk proportions similar in each fold.

For Logistic Regression and SVM, scaling is placed inside a `make_pipeline()` object. This avoids data leakage because the scaler is refit separately inside each cross-validation training fold instead of learning from the whole dataset at once.


## ROC curves comparison

**Purpose**: Visualize one-vs-rest ROC curves for each risk category using Random Forest probabilities. This provides a class-by-class view of how well the model separates each label from the others.

**Input**: `X_test`, `y_test`, the trained `rf_model`, and the fitted `LabelEncoder`.

**Output**: ROC curves and AUC values for classes where the test set contains both positive and negative examples. If a class cannot produce a valid ROC curve, the notebook reports that instead of failing.


In [ ]:
# ROC Curves for each class using the Random Forest model
classes = rf_model.classes_
y_test_bin = label_binarize(y_test, classes=classes)
y_score = rf_model.predict_proba(X_test)

plt.figure(figsize=(10, 8))
colors = ['blue', 'red', 'green', 'purple', 'orange']
valid_curve_count = 0

for idx, class_id in enumerate(classes):
    class_name = le_target.inverse_transform([class_id])[0]
    y_binary = y_test_bin[:, idx]

    if len(np.unique(y_binary)) < 2:
        print(f"Skipping ROC curve for {class_name}: test set does not contain both positive and negative examples.")
        continue

    fpr, tpr, _ = roc_curve(y_binary, y_score[:, idx])
    class_auc = roc_auc_score(y_binary, y_score[:, idx])
    plt.plot(
        fpr,
        tpr,
        color=colors[idx % len(colors)],
        lw=2,
        label=f'{class_name} (AUC = {class_auc:.2f})'
    )
    valid_curve_count += 1

if valid_curve_count > 0:
    plt.plot([0, 1], [0, 1], 'k--', lw=2)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - Random Forest (One-vs-Rest)')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
else:
    plt.close()
    print('No valid ROC curves could be plotted for this test split.')


### Code explanation - ROC curves

ROC curves require binary labels, so the code converts the multiclass target into one-vs-rest format with `label_binarize()`. For each class, the curve compares the true-positive rate against the false-positive rate at different probability thresholds.

The guard clause checks whether a class has both positive and negative examples in the test set. This is necessary because the dataset is tiny; without both sides of a class, a valid ROC curve cannot be computed.


### Final project interpretation

**Purpose**: Convert the metric tables and plots into final-project conclusions. This section identifies the strongest model in the demonstration, summarizes important predictors, and states the limitations clearly.

**Input**: `results`, `model_report_summary`, `cv_results`, `importances`, and `top_corr`.

**Output**: A final written-style summary that can be used as the notebook's conclusion.


In [ ]:
# --- FINAL PROJECT INTERPRETATION ---
best_test_model = results.sort_values('Accuracy', ascending=False).iloc[0]
best_macro_f1_model = model_report_summary.sort_values('Macro F1', ascending=False).iloc[0]
best_cv_model = cv_results.sort_values('Mean CV Accuracy', ascending=False).iloc[0]

top_importance = importances.head(5).rename('Random Forest Importance').to_frame()
top_correlation = top_corr.iloc[1:6].rename('Absolute Correlation with Next-Year Deaths').to_frame()

print('=' * 70)
print('FINAL PROJECT INTERPRETATION')
print('=' * 70)
print()
print(f"Best model by single test Accuracy: {best_test_model['Model']} ({best_test_model['Accuracy']:.4f})")
print(f"Best model by test Macro F1: {best_macro_f1_model['Model']} ({best_macro_f1_model['Macro F1']:.4f})")
print(f"Best model by 5-fold CV Accuracy: {best_cv_model['Model']} ({best_cv_model['Mean CV Accuracy']:.4f} ± {best_cv_model['Std CV Accuracy']:.4f})")
print()
print('Top Random Forest feature importances:')
display(top_importance.round(4))
print('Top absolute correlations with next-year deaths:')
display(top_correlation.round(4))
print()
print('Conclusion:')
print('This notebook is a complete demonstration of a death-risk classification pipeline. It imports risk-factor features, aligns Excel-backed registered death targets, creates future-shifted risk categories, trains four classifiers, and evaluates them with test metrics, classification reports, confusion matrices, ROC curves, feature analysis, and cross-validation.')
print()
print('Main limitation:')
print('The dataset is very small and partially uses extrapolated/provisional values, so the results are suitable for a final-project demonstration but not for real-world mortality-risk deployment.')


## Prediction function

**Purpose**: Provide a reusable helper that accepts a dictionary of risk-factor values and returns a predicted risk category with class probabilities.

**Input**: A year label for reference, a dictionary whose keys match entries in `feature_cols`, and a trained classifier. Missing risk factors default to zero, so predictions are most meaningful when the caller supplies a complete feature set.

**Output**: A dictionary containing the reference year, predicted category, and probability assigned to each risk class.


In [ ]:
def predict_death_risk(year, risk_factors_dict, model=rf_model):
    """
    Predict the death-risk category for a supplied risk-factor profile.

    Parameters
    ----------
    year : int
        Reference year for the prediction output.
    risk_factors_dict : dict
        Dictionary of risk-factor values. Keys should match feature_cols.
    model : sklearn classifier
        Trained classifier to use for prediction. Defaults to Random Forest.

    Returns
    -------
    dict
        Predicted category and class probabilities.
    """
    input_data = {col: 0 for col in feature_cols}

    for factor, value in risk_factors_dict.items():
        if factor in input_data:
            input_data[factor] = value

    input_df = pd.DataFrame([input_data], columns=feature_cols)
    prediction = model.predict(input_df)[0]
    probabilities = model.predict_proba(input_df)[0]

    return {
        'Year': year,
        'Predicted Category': le_target.inverse_transform([prediction])[0],
        'Probabilities': {
            le_target.classes_[i]: prob for i, prob in enumerate(probabilities)
        }
    }

print('Example Prediction Function:')
print('Use predict_death_risk(year, risk_factors_dict) with values for the risk-factor columns.')


### Code explanation - prediction helper

`predict_death_risk()` is a reusable wrapper around the trained model. It creates a one-row DataFrame with the same columns as `feature_cols`, fills in any provided risk-factor values, and asks the model for a predicted risk class plus class probabilities.

Missing inputs default to zero, so this function is mainly a demonstration of how prediction would work. A realistic use should supply a complete risk-factor profile using the same feature definitions and scale as the training data.


### Phase 3 - Summary and conclusions

**Purpose**: Summarize the modeling workflow, clarify how the imported death-count targets should be interpreted, and explain why the reported metrics should be treated carefully.

**Input**: The prepared dataset, the Excel-backed target sources, trained model outputs, and the comparison table.

**Output**: A concise conclusion block covering target interpretation, micro-dataset limits, models evaluated, and practical recommendations.

In [ ]:
print('=' * 60)
print('DEATH RISK PREDICTION - MODEL COMPARISON SUMMARY')
print('=' * 60)
print()
print('Note on Target Variable Interpretation:')
print('Risk-factor columns are the model inputs. Registered death counts imported from PSA Excel workbooks define the recent target values used to create Low, Medium, and High Risk labels.')
print('The 2025 target is provisional and annualized from Jan-Oct data for demonstration, so it should not be treated as a finalized official full-year total.')
print()
print('Target Source Coverage:')
print(df[['Year', 'Total_Deaths_Real', 'Death_Target_Source', 'Next_Year_Target', 'Next_Year_Target_Source']].tail(12).to_string(index=False))
print()
print('Final Model Takeaway:')
print(f"Best single-split Accuracy: {best_test_model['Model']} ({best_test_model['Accuracy']:.4f})")
print(f"Best Macro F1: {best_macro_f1_model['Model']} ({best_macro_f1_model['Macro F1']:.4f})")
print(f"Best cross-validation Accuracy: {best_cv_model['Model']} ({best_cv_model['Mean CV Accuracy']:.4f} ± {best_cv_model['Std CV Accuracy']:.4f})")
print()
print('Note on Micro-Dataset Limits:')
print(f'The working dataset contains only {len(df)} rows after target shifting. With a test set of {len(y_test)} rows, accuracy and ROC AUC can move sharply from just one changed prediction.')
print()
print('Models Evaluated:')
print('1. Logistic Regression - simple baseline multiclass classifier')
print('2. Support Vector Machine - nonlinear classifier with hyperparameter tuning')
print('3. Decision Tree - interpretable tree-based model with capped depth')
print('4. Random Forest - capped ensemble of decision trees')
print()
print('Recommendations:')
print('- Treat this notebook as a polished final-project demonstration, not a deployable mortality-risk model.')
print('- Prefer the model that performs consistently across test metrics and cross-validation, not only the one with the highest single-split score.')
print('- For real forecasting, use validated full-year death counts, richer external predictors, and chronological validation before trusting out-of-sample performance.')


### How to read this output - final summary

The summary cell prints the target interpretation, recent target-source coverage, model list, and limitations. The source table is especially important because it shows which target values came from PSA Excel workbooks and which earlier values are proxy totals.

This final explanation supports the main idea of the notebook: the models are predicting relative death-risk categories from risk-factor inputs, using shifted registered-death counts as the target signal.


**Note on results:** Because the dataset is extremely small after shifting, the evaluation metrics can be unstable. The results should be interpreted as a demonstration of a classification pipeline, not as a rigorous statistical analysis.
